# 🐱🐶 Image Classification: CNN → Overfitting Fixes → Transfer Learning

**A binary image classifier (Cats vs Dogs) built three ways, compared head-to-head:**

1. **Baseline CNN** — a small convnet trained from scratch (overfits on purpose, so we can *see* it happen)
2. **Improved CNN** — the same architecture, fixed with data augmentation + dropout + smart callbacks
3. **Transfer Learning** — EfficientNetB0 pretrained on ImageNet, feature-extraction + fine-tuning

**Stack:** `tf.keras` (TensorFlow), `kagglehub` for the dataset, scikit-learn for evaluation metrics.

**Dataset:** [Cats and Dogs](https://www.kaggle.com/datasets/marquis03/cats-and-dogs) (Kaggle) — a deliberately **lightweight** binary classification dataset (~10.3 MB total, 275 train + 70 validation images), downloaded via `kagglehub`. Small enough to download in seconds and train fully on CPU in a few minutes, while still being large enough to clearly demonstrate overfitting and its fixes.

> **How to run:** Open in Google Colab, set Runtime → GPU (optional but faster), run top to bottom.
> You'll need a Kaggle account + API token for `kagglehub` to authenticate (it will prompt you).

---


## 1. Setup & Reproducibility

Install/import dependencies and fix every random seed we can reach so results are repeatable.

In [ ]:
# If running in Colab, uncomment to make sure kagglehub is available:
# !pip install -q kagglehub

import os
import random
import shutil
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 2. Dataset Download (`kagglehub`)

Download the **Cats and Dogs** dataset manually from [kaggle.com/datasets/marquis03/cats-and-dogs](https://www.kaggle.com/datasets/marquis03/cats-and-dogs) (only ~10.3 MB), extract it, and point `LOCAL_DATASET_PATH` at the extracted folder below — no Kaggle API credentials needed. (`kagglehub` auto-download is available as a fallback if you'd rather set that up instead.) `kagglehub` caches the download, so
re-running this cell is instant after the first time. Because different repackagings of this dataset
use slightly different folder layouts, we **scan the downloaded path** for images instead of hardcoding
folder names — this makes the notebook robust to whichever version gets downloaded.

In [ ]:
# --- Dataset source: local folder (manual download) OR kagglehub (automatic) ---
# Manual download (what we'll use here):
#   1. Go to: https://www.kaggle.com/datasets/marquis03/cats-and-dogs
#   2. Click "Download" (top right) -> downloads a small "archive.zip" (~10.3MB)
#   3. Extract it anywhere on your machine
#   4. Paste the extracted folder's path below (Windows example given)
#
# If LOCAL_DATASET_PATH points to a real folder, we use it directly and skip
# kagglehub entirely. Leave it as "" to fall back to kagglehub.dataset_download()
# (requires Kaggle API credentials — see the earlier discussion if you want that instead).

LOCAL_DATASET_PATH = r"C:\Users\Swagat\Downloads\CatDog"

if LOCAL_DATASET_PATH and pathlib.Path(LOCAL_DATASET_PATH).exists():
    dataset_root = LOCAL_DATASET_PATH
    print("Using local dataset at:", dataset_root)
else:
    import kagglehub
    DATASET_SLUG = "marquis03/cats-and-dogs"
    dataset_root = kagglehub.dataset_download(DATASET_SLUG)
    print("Dataset downloaded to:", dataset_root)

# Peek at the top-level structure
for p in sorted(pathlib.Path(dataset_root).rglob("*"))[:20]:
    print(p.relative_to(dataset_root))


In [ ]:
# --- Robustly discover every (image_path, label) pair -----------------------
# The competition's naming convention labels files as "cat.123.jpg" / "dog.456.jpg",
# and most repackagings also keep "cat"/"dog" folder names. We check both.

IMG_EXTS = {".jpg", ".jpeg", ".png"}

def infer_label(path: pathlib.Path):
    name = path.name.lower()
    parent = path.parent.name.lower()
    if "cat" in name or "cat" in parent:
        return "cat"
    if "dog" in name or "dog" in parent:
        return "dog"
    return None

all_files = [p for p in pathlib.Path(dataset_root).rglob("*") if p.suffix.lower() in IMG_EXTS]
records = []
for p in all_files:
    label = infer_label(p)
    if label is not None:
        records.append((str(p), label))

df_all = pd.DataFrame(records, columns=["filepath", "label"])
print("Total labeled images found:", len(df_all))
print(df_all["label"].value_counts())


In [ ]:
# --- Use the full (already-small) dataset, balanced across classes ----------
# This dataset is tiny by design (~10.3MB total), so instead of down-sampling
# further we just balance classes to whichever class has fewer images.
# MAX_PER_CLASS is a safety cap in case a much bigger dataset slug is swapped in later.

MAX_PER_CLASS = 500
class_counts = df_all["label"].value_counts()
n_per_class = min(class_counts.min(), MAX_PER_CLASS)
print(f"Available per class: {dict(class_counts)}  ->  using {n_per_class} per class")

set_seeds(SEED)
df_cats = df_all[df_all.label == "cat"].sample(n=n_per_class, random_state=SEED)
df_dogs = df_all[df_all.label == "dog"].sample(n=n_per_class, random_state=SEED)
df_subset = pd.concat([df_cats, df_dogs]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Subset size:", len(df_subset))
df_subset["label"].value_counts()


In [ ]:
# --- Stratified train / val / test split (70 / 15 / 15) --------------------
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_subset, test_size=0.30, stratify=df_subset["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, dict(d["label"].value_counts()))


In [ ]:
# --- Materialize the split as an on-disk directory tree ---------------------
# image_dataset_from_directory expects: root/split/class_name/*.jpg
# We copy (not move) the small subset so the original kagglehub cache is untouched.

DATA_ROOT = pathlib.Path("/content/data") if pathlib.Path("/content").exists() else pathlib.Path("./data")
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for label in ["cat", "dog"]:
        (DATA_ROOT / split_name / label).mkdir(parents=True, exist_ok=True)
    for _, row in split_df.iterrows():
        src = pathlib.Path(row["filepath"])
        dst = DATA_ROOT / split_name / row["label"] / src.name
        if not dst.exists():
            shutil.copy(src, dst)

print("Directory tree ready at:", DATA_ROOT.resolve())


## 3. Data Exploration

Sanity-check class balance and look at real samples before writing a single line of model code.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (split_name, split_df) in zip(axes, [("Train", train_df), ("Val", val_df)]):
    split_df["label"].value_counts().plot(kind="bar", ax=ax, color=["#f0a", "#0af"])
    ax.set_title(f"{split_name} class balance")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
def show_samples(df, n=8, title=""):
    sample = df.sample(n=n, random_state=SEED)
    plt.figure(figsize=(14, 4))
    for i, (_, row) in enumerate(sample.iterrows()):
        img = keras.utils.load_img(row["filepath"])
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(img)
        plt.title(row["label"])
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(train_df, title="Sample training images")


In [ ]:
# Image size sanity check — helps decide our target input size
sizes = [keras.utils.load_img(p).size for p in train_df["filepath"].sample(30, random_state=SEED)]
widths, heights = zip(*sizes)
print(f"Width  range: {min(widths)}–{max(widths)} (median {int(np.median(widths))})")
print(f"Height range: {min(heights)}–{max(heights)} (median {int(np.median(heights))})")


## 4. Building `tf.data` Pipelines

We resize everything to a common size, batch it, and use `.cache()` / `.prefetch()` for speed.
Pixel rescaling (`1/255`) is done **inside the model** as a `Rescaling` layer, so the same
saved model works correctly on raw images at inference time — no separate preprocessing script needed.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    DATA_ROOT / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED
)
val_ds = keras.utils.image_dataset_from_directory(
    DATA_ROOT / "val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED
)
test_ds = keras.utils.image_dataset_from_directory(
    DATA_ROOT / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)  # ['cat', 'dog'] -> label 0 = cat, 1 = dog

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 5. Baseline CNN (Deliberately Overfit-Prone)

A small convnet with **no** dropout and **no** augmentation. With only ~1,700 training images and
several thousand trainable parameters per conv block, this model has more than enough capacity to
start memorizing the training set — which is exactly the failure mode we want to demonstrate.

In [ ]:
def build_baseline_cnn(input_shape=(160, 160, 3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1.0 / 255)(inputs)

    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="baseline_cnn")

set_seeds(SEED)
baseline_model = build_baseline_cnn()
baseline_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
baseline_model.summary()


In [ ]:
EPOCHS_BASELINE = 15

baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_BASELINE,
    verbose=2,
)


## 6. Learning Curves & Overfitting Diagnosis

In [ ]:
def plot_history(history, title):
    hist = history.history
    epochs_range = range(1, len(hist["loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs_range, hist["accuracy"], label="train")
    axes[0].plot(epochs_range, hist["val_accuracy"], label="val")
    axes[0].set_title(f"{title} — Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(epochs_range, hist["loss"], label="train")
    axes[1].plot(epochs_range, hist["val_loss"], label="val")
    axes[1].set_title(f"{title} — Loss")
    axes[1].set_xlabel("Epoch"); axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(baseline_history, "Baseline CNN")


**What to look for:** as training progresses, the **training accuracy climbs toward ~100%** while
**validation accuracy plateaus (or gets noisy/worse)**, and the **gap between training and validation
loss widens** — training loss keeps dropping, validation loss flattens out or starts creeping back up.

That divergence is the textbook signature of **overfitting**: the network has enough capacity (four
conv blocks + a 256-unit dense layer) and too little data/regularization, so past a certain point it
starts memorizing training-set idiosyncrasies (exact backgrounds, lighting, poses) instead of learning
features that generalize to unseen cats and dogs.

**Fixes we'll apply next:**
- **Data augmentation** — synthetically expands the training distribution (random flips/rotations/zooms),
  making it harder to memorize exact pixels.
- **Dropout** — randomly zeroes activations during training, discouraging co-adapted features.
- **Callbacks** — `EarlyStopping` stops us at the best validation point instead of overfitting further,
  `ModelCheckpoint` saves that best point, and `ReduceLROnPlateau` shrinks the learning rate when
  progress stalls.

## 7. Improved CNN — Data Augmentation + Dropout + Callbacks

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# Preview a few augmented versions of the same image
sample_batch = next(iter(train_ds.take(1)))[0][:1]
plt.figure(figsize=(12, 3))
for i in range(6):
    augmented = data_augmentation(sample_batch, training=True)
    plt.subplot(1, 6, i + 1)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle("Augmentation preview (same source image)")
plt.tight_layout()
plt.show()


In [ ]:
def build_improved_cnn(input_shape=(160, 160, 3)):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="improved_cnn")

set_seeds(SEED)
improved_model = build_improved_cnn()
improved_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
improved_model.summary()


In [ ]:
os.makedirs("checkpoints", exist_ok=True)
IMPROVED_CKPT_PATH = "checkpoints/improved_cnn_best.keras"

improved_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(IMPROVED_CKPT_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
]

EPOCHS_IMPROVED = 30  # EarlyStopping will typically stop well before this

improved_history = improved_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_IMPROVED,
    callbacks=improved_callbacks,
    verbose=2,
)


In [ ]:
plot_history(improved_history, "Improved CNN")

**Result:** the train/validation curves should now track each other much more closely — the gap
that appeared in the baseline is visibly narrower, and validation loss no longer diverges upward the
same way. That's augmentation + dropout doing their job, with `EarlyStopping` making sure we keep the
best-performing weights rather than whatever the last epoch happened to produce.

## 8. Save & Reload the Best Model

Confirm the checkpoint saved by `ModelCheckpoint` reloads cleanly and reproduces the same validation performance.

In [ ]:
reloaded_improved_model = keras.models.load_model(IMPROVED_CKPT_PATH)

original_val_loss, original_val_acc = improved_model.evaluate(val_ds, verbose=0)
reloaded_val_loss, reloaded_val_acc = reloaded_improved_model.evaluate(val_ds, verbose=0)

print(f"In-memory model   -> val_loss: {original_val_loss:.4f}, val_acc: {original_val_acc:.4f}")
print(f"Reloaded model    -> val_loss: {reloaded_val_loss:.4f}, val_acc: {reloaded_val_acc:.4f}")
assert abs(original_val_acc - reloaded_val_acc) < 1e-3, "Reloaded model doesn't match saved weights!"
print("✅ Reload verified — saved checkpoint matches training-time performance.")


## 9. Transfer Learning with EfficientNetB0

Instead of learning visual features from scratch, we start from **EfficientNetB0** pretrained on
ImageNet (1.4M images, 1000 classes — including many dog/cat breeds). We freeze its convolutional
base and train only a small classification head on top ("feature extraction"), then unfreeze the
top layers for a brief **fine-tuning** pass at a very low learning rate.

In [ ]:
EFF_IMG_SIZE = (160, 160)  # EfficientNetB0 accepts variable input sizes >= 32x32

base_model = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=EFF_IMG_SIZE + (3,)
)
base_model.trainable = False  # freeze for feature-extraction phase

inputs = keras.Input(shape=EFF_IMG_SIZE + (3,))
x = data_augmentation(inputs)
# Note: EfficientNet's preprocess_input expects raw 0-255 pixel values, NOT [0,1] floats.
x = keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

effnet_model = keras.Model(inputs, outputs, name="efficientnet_transfer")
effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
effnet_model.summary()


In [ ]:
EFFNET_CKPT_PATH = "checkpoints/effnet_best.keras"

effnet_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(EFFNET_CKPT_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

EPOCHS_FEATURE_EXTRACT = 10

effnet_history = effnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FEATURE_EXTRACT,
    callbacks=effnet_callbacks,
    verbose=2,
)

plot_history(effnet_history, "EfficientNetB0 (feature extraction)")


### Fine-tuning

Now we unfreeze the **top ~20%** of the base model's layers and continue training at a much lower
learning rate. This lets the highest-level features adapt slightly to cats/dogs specifically, without
destroying the low-level ImageNet features (edges, textures) or blowing up the gradients.

In [ ]:
base_model.trainable = True

# Freeze everything except roughly the last fifth of layers
fine_tune_at = int(len(base_model.layers) * 0.8)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_FINE_TUNE = 5
total_epochs_so_far = len(effnet_history.history["loss"])

fine_tune_history = effnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs_so_far + EPOCHS_FINE_TUNE,
    initial_epoch=total_epochs_so_far,
    callbacks=effnet_callbacks,
    verbose=2,
)


In [ ]:
# Stitch feature-extraction + fine-tuning histories together for one combined curve
combined_history = {
    k: effnet_history.history[k] + fine_tune_history.history[k] for k in effnet_history.history
}

class _H:  # tiny shim so plot_history() works unchanged on the combined dict
    def __init__(self, h): self.history = h

plot_history(_H(combined_history), "EfficientNetB0 (feature extraction → fine-tuning)")
plt.axvline(x=EPOCHS_FEATURE_EXTRACT, color="gray", linestyle="--", alpha=0.6)


In [ ]:
reloaded_effnet_model = keras.models.load_model(EFFNET_CKPT_PATH)
print("Best EfficientNet checkpoint reloaded from:", EFFNET_CKPT_PATH)


## 10. Test-Set Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

models_to_eval = {
    "Baseline CNN": baseline_model,
    "Improved CNN": reloaded_improved_model,
    "EfficientNetB0": reloaded_effnet_model,
}

test_results = {}
for name, model in models_to_eval.items():
    loss, acc = model.evaluate(test_ds, verbose=0)
    test_results[name] = {"loss": loss, "accuracy": acc}
    print(f"{name:16s} -> test_loss: {loss:.4f}, test_accuracy: {acc:.4f}")


In [ ]:
# Confusion matrix + classification report for our best model (EfficientNet)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_proba = reloaded_effnet_model.predict(test_ds, verbose=0).ravel()
y_pred = (y_pred_proba > 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap="Blues")
plt.title("EfficientNetB0 — Confusion Matrix (test set)")
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))


## 11. Sample Predictions

In [ ]:
# Grab one batch of test images (unshuffled, so it lines up with y_true/y_pred order for batch 0)
images_batch, labels_batch = next(iter(test_ds.take(1)))
preds_batch = reloaded_effnet_model.predict(images_batch, verbose=0).ravel()

plt.figure(figsize=(14, 7))
for i in range(min(12, len(images_batch))):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(images_batch[i].numpy().astype("uint8"))
    true_label = class_names[int(labels_batch[i])]
    pred_label = class_names[int(preds_batch[i] > 0.5)]
    confidence = preds_batch[i] if pred_label == "dog" else 1 - preds_batch[i]
    color = "green" if true_label == pred_label else "red"
    plt.title(f"true: {true_label} | pred: {pred_label} ({confidence:.0%})", color=color, fontsize=9)
    plt.axis("off")
plt.suptitle("Sample test predictions (EfficientNetB0)")
plt.tight_layout()
plt.show()


## 12. Final Comparison: Baseline CNN vs Improved CNN vs EfficientNetB0

In [ ]:
comparison_df = pd.DataFrame(test_results).T
comparison_df["accuracy"] = (comparison_df["accuracy"] * 100).round(2)
comparison_df["loss"] = comparison_df["loss"].round(4)
comparison_df.columns = ["Test Loss", "Test Accuracy (%)"]
comparison_df = comparison_df.sort_values("Test Accuracy (%)")
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(comparison_df.index, comparison_df["Test Accuracy (%)"], color=["#e74c3c", "#f39c12", "#27ae60"])
ax.set_xlabel("Test Accuracy (%)")
ax.set_xlim(0, 100)
ax.set_title("Model Comparison — Test Accuracy")
for bar, val in zip(bars, comparison_df["Test Accuracy (%)"]):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center")
plt.tight_layout()
plt.show()


## 13. Conclusions

- **Baseline CNN** trained from scratch on ~1,700 images overfit quickly and predictably: training
  accuracy kept climbing toward 100% while validation accuracy plateaued and the train/val loss gap
  widened — the classic overfitting signature for a high-capacity model on a small dataset.
- **Data augmentation + dropout + callbacks** closed most of that gap. Augmentation effectively grew
  the training distribution, dropout discouraged over-reliant feature co-adaptation, and
  `EarlyStopping`/`ModelCheckpoint` guaranteed we kept the best-generalizing weights rather than the
  most-overfit ones. This is the standard, cheap first line of defense against overfitting on small
  image datasets — no extra data or compute required.
- **Transfer learning with EfficientNetB0** outperformed both from-scratch CNNs by a wide margin,
  despite seeing the *same* small training set. Features learned from 1.4M ImageNet images generalize
  well to cats/dogs, and fine-tuning the top layers squeezed out a further improvement without needing
  more data. This is why transfer learning is the default starting point for most real-world image
  classification tasks, especially with limited labeled data.
- **Practical takeaway:** when data is scarce, prioritize (1) transfer learning, then (2) augmentation
  + regularization + early stopping — in that order — before considering collecting more data or
  building a bigger model from scratch.

**Possible extensions:** train on the full 25k-image dataset, try a larger EfficientNet variant,
add test-time augmentation, or deploy the saved `.keras` model behind a small Gradio/Streamlit demo.
